# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client:
 Version:    29.8.1
 Context:    default
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.141.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /usr/local/lib/docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /usr/local/lib/docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /usr/local/lib/docker/cli-plugins/docker-dh

### What is a container?

A container is an isolated environment that packages an application together with all dependencies needed to run it. This allows the application to run consistently on different systems without requiring the software and its dependencies to be installed directly on the host system.

### Why do we use containers?

We use containers to make software portable and reproducible. A container provides the same software and dependencies regardless of the system on which it is executed. This avoids problems caused by different software versions or missing dependencies and makes computational workflows easier to reproduce and share.

### What is a docker image?

A Docker image is a read-only template that contains an application and all dependencies needed to run it. Docker uses an image to create and run containers. Multiple containers can be created from the same image.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly

!docker login

### Run your first docker container

In [ ]:
!docker run hello-world

Hello from Docker!
This message shows that your installation appears to be working correctly.

### Find the container ID

In [ ]:
!docker ps -a # output: f5d8700a2015

### Delete the container again, give prove its deleted

In [ ]:
!docker rm f5d8700a2015 # docker returns the container ID of the container we just deleted.
!docker ps -a # shows us all docker containers, excluding the one we just deleted.

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. I installed FastQC using the Ubuntu package manager with `sudo apt install fastqc`. The required dependencies, including Java, were installed automatically.

2. I checked the installed FastQC version using `fastqc --version`. The installed version was FastQC v0.12.1.

3. I ran FastQC on the example FASTQ file from last week using:
   `fastqc day_02/fetchngs_results/fastq/SRX19144486_SRR23195516_1.fastq.gz`

4. FastQC analyzed the reads and generated an HTML report and a ZIP file containing the results.

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container

!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

In [ ]:
# run the container and save the results to a new "fastqc_results" directory

!docker run --rm \
    -v "$PWD:/data" \
    community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
    fastqc \
    -o /data/day_04/fastqc_results \
    /data/day_02/fetchngs_results/fastq/SRX19144486_SRR23195516_1.fastq.gz

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

Installing FastQC directly was straightforward, but it also required installing the software and its dependencies on the host system. Using a Docker container should be easier in the future because the required software and dependencies are already included in the image and do not have to be installed separately.

### What would you say, which approach is more reproducible?

The Docker approach is more reproducible because the container provides a defined software environment with specific versions of the software and its dependencies. The same container can therefore be used on different systems with the same environment.

### Compare the file to last weeks fastqc results, are they identical?

Yes. The FastQC results are identical. The local FastQC run, the Docker FastQC run, and last week's FastQC results report the same QC results. For example, they report 20,677,059 reads, a sequence length of 150 bp, and 48% GC content. MultiQC visualizes the results differently, but the underlying FastQC results are the same.

### Is the fastqc version identical?

Yes. The local FastQC installation, the Docker container, and last week's nf-core/rnaseq pipeline all used FastQC v0.12.1.


## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor

!nano day_04/my_dockerfile

### Explain the RUN and ENV lines you added to the file

The `RUN` line updates the Debian package list and installs `curl` and `cowsay` inside the image.

The `ENV` line adds `/usr/games` to the `PATH` environment variable. This is necessary because `cowsay` is installed in `/usr/games`, which is not included in the default PATH of the base image. Adding it allows `cowsay` to be called directly without specifying its full path.

In [ ]:
# build the docker image

!docker --config "$PWD/config/docker-clean" build -t my-cowsay -f day_04/my_dockerfile .

In [ ]:
# make sure that the image has been built

!docker images my-cowsay

In [ ]:
# run the docker file 

!docker run --rm my-cowsay cowsay "welcome to the world of docker"

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

!cat day_04/salmon_docker

In [ ]:
# build the image

!docker --config "$PWD/config/docker-clean" build -t my-salmon -f day_04/salmon_docker .


In [ ]:
# run the docker image to give out the version of salmon

!docker run --rm my-salmon salmon --version


## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

No, bioinformaticians do not have to create a new Docker image every time they want to use a tool. Existing container images can be obtained from public container registries.

---

Find the salmon docker image online and run it on your computer.


In [ ]:
!docker run --rm \
    community.wave.seqera.io/library/salmon:1.5.2--h84f40af_0 \
    salmon --version

---

What is https://biocontainers.pro/ ?

BioContainers is a community project that provides ready-to-use container images for bioinformatics software. These containers include the software and its dependencies, making tools easier to install and analyses more reproducible.

## Are there other ways to create Docker (or Apptainer) images?

Yes. Container images can also be generated automatically instead of writing a Dockerfile manually.

What is https://seqera.io/containers/ ?

Seqera Containers is a free service that creates container images on demand from Conda or PyPI packages. Users can select the required packages and generate Docker or Singularity containers, which makes creating reproducible software environments easier.